# Advanced Observability & Distributed Tracing Guide

**Complete guide for distributed tracing, metrics collection, log aggregation, and OpenTelemetry integration.**

Learn how to observe complex systems across multiple services.


## Observability Framework

### Three Pillars

```
Observability (Understanding system behavior)
├─ Logs (What happened?)
│  ├─ Application logs
│  ├─ Access logs
│  └─ Structured logs (JSON)
│
├─ Metrics (How much?)
│  ├─ Counters (total count)
│  ├─ Gauges (current value)
│  ├─ Histograms (distribution)
│  └─ Summaries (percentiles)
│
└─ Traces (How did it happen?)
   ├─ Request traces
   ├─ Span relationships
   ├─ Timing information
   └─ Error tracking
```

### Observability vs Monitoring

```
Monitoring: Alert when things go wrong
└─ Pre-defined metrics
└─ Known unknowns

Observability: Understand WHY things went wrong
└─ Any query on any data
└─ Unknown unknowns
```

---

## Distributed Tracing with OpenTelemetry

### Trace Architecture

```python
# shared/tracing.py
from opentelemetry import trace, metrics
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.jaeger.thrift import JaegerExporter
from opentelemetry.sdk.resources import SERVICE_NAME, Resource
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor
from opentelemetry.instrumentation.sqlalchemy import SQLAlchemyInstrumentor
from opentelemetry.instrumentation.requests import RequestsInstrumentor
from opentelemetry.instrumentation.psycopg2 import Psycopg2Instrumentor

def init_tracing(service_name: str):
    """Initialize OpenTelemetry tracing."""

    # Create Jaeger exporter
    jaeger_exporter = JaegerExporter(
        agent_host_name=os.getenv("JAEGER_AGENT_HOST", "localhost"),
        agent_port=int(os.getenv("JAEGER_AGENT_PORT", 6831)),
    )

    # Create trace provider
    trace_provider = TracerProvider(
        resource=Resource.create({SERVICE_NAME: service_name})
    )
    trace_provider.add_span_processor(BatchSpanProcessor(jaeger_exporter))

    # Set global trace provider
    trace.set_tracer_provider(trace_provider)

    # Auto-instrument common libraries
    FastAPIInstrumentor.instrument_app(app)
    SQLAlchemyInstrumentor().instrument(engine=db.engine)
    RequestsInstrumentor().instrument()
    Psycopg2Instrumentor().instrument()

    return trace.get_tracer(__name__)

# Usage
tracer = init_tracing("aria-api")

@app.route("/api/chat", methods=["POST"])
async def chat(req: func.HttpRequest):
    """Traced endpoint."""
    with tracer.start_as_current_span("chat_request") as span:
        span.set_attribute("user_id", user_id)
        span.set_attribute("model", model)

        with tracer.start_as_current_span("validate_input"):
            # Validation logic
            pass

        with tracer.start_as_current_span("fetch_context"):
            # Memory fetch
            pass

        with tracer.start_as_current_span("generate_response"):
            # LLM call
            pass

        return response
```

### Custom Spans

```python
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode

tracer = trace.get_tracer(__name__)

def process_complex_workflow():
    """Trace a complex multi-step workflow."""

    # Root span
    with tracer.start_as_current_span("complex_workflow") as root_span:
        root_span.set_attribute("workflow_id", "abc123")

        # Child span 1: Data collection
        with tracer.start_as_current_span("collect_data") as span:
            span.set_attribute("source", "database")
            try:
                data = fetch_data()
                span.set_attribute("rows_fetched", len(data))
            except Exception as e:
                span.set_attribute("error", True)
                span.set_attribute("error_type", type(e).__name__)
                span.record_exception(e)
                span.set_status(Status(StatusCode.ERROR))
                raise

        # Child span 2: Processing
        with tracer.start_as_current_span("process_data") as span:
            processed = process_data(data)
            span.set_attribute("output_size", len(processed))

        # Child span 3: Storage
        with tracer.start_as_current_span("store_results") as span:
            store_results(processed)
            span.add_event("results_stored")

        root_span.set_status(Status(StatusCode.OK))
```

---

## Metrics Collection

### Key Metrics

```python
# shared/metrics.py
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.prometheus import PrometheusMetricReader

# Initialize metrics
prometheus_reader = PrometheusMetricReader()
meter_provider = MeterProvider(metric_readers=[prometheus_reader])
metrics.set_meter_provider(meter_provider)
meter = metrics.get_meter(__name__)

# Define metrics
request_counter = meter.create_counter(
    "http.requests",
    description="Total HTTP requests",
    unit="requests"
)

request_duration = meter.create_histogram(
    "http.request.duration",
    description="HTTP request duration",
    unit="ms"
)

active_connections = meter.create_observable_gauge(
    "db.connections.active",
    description="Active database connections",
    unit="connections",
    callbacks=[lambda obs: obs.observe(get_active_connections())]
)

error_rate = meter.create_counter(
    "errors.total",
    description="Total errors",
    unit="errors"
)

# Usage in request handler
@app.route("/api/chat", methods=["POST"])
async def chat(req: func.HttpRequest):
    start_time = time.time()

    try:
        # Process request
        response = await process_chat(req)

        # Record metrics
        duration_ms = (time.time() - start_time) * 1000
        request_duration.record(duration_ms, {"endpoint": "/api/chat"})
        request_counter.add(1, {"status": "success"})

        return response

    except Exception as e:
        error_rate.add(1, {"error_type": type(e).__name__})
        request_counter.add(1, {"status": "error"})
        raise
```

### Key Metrics to Track

```yaml
API Performance:
    - Request rate (req/sec)
    - Response time (p50, p95, p99)
    - Error rate (%)
    - Status codes (2xx, 4xx, 5xx)

Database:
    - Query latency (p50, p95, p99)
    - Connection pool size
    - Active connections
    - Query count by type

LLM/ML:
    - Tokens generated (per second)
    - Model response time
    - Completion success rate
    - Token cost per request

Business:
    - Active users (DAU/MAU)
    - Feature usage
    - Conversion rate
    - Revenue per user

Infrastructure:
    - CPU usage (%)
    - Memory usage (%)
    - Disk I/O (bytes/sec)
    - Network I/O (bytes/sec)
```

---

## Structured Logging

### JSON Logging

```python
# shared/logging_config.py
import json
import logging
from datetime import datetime

class JSONFormatter(logging.Formatter):
    """Format logs as JSON for easy parsing."""

    def format(self, record: logging.LogRecord) -> str:
        log_data = {
            "timestamp": datetime.utcnow().isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
            "module": record.module,
            "function": record.funcName,
            "line": record.lineno,
        }

        # Add exception info if present
        if record.exc_info:
            log_data["exception"] = self.formatException(record.exc_info)

        # Add custom fields from extra
        if hasattr(record, "user_id"):
            log_data["user_id"] = record.user_id
        if hasattr(record, "request_id"):
            log_data["request_id"] = record.request_id
        if hasattr(record, "duration_ms"):
            log_data["duration_ms"] = record.duration_ms

        return json.dumps(log_data)

# Configure logging
def setup_logging():
    logger = logging.getLogger()
    handler = logging.StreamHandler()
    formatter = JSONFormatter()
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

# Usage
logger = logging.getLogger(__name__)

def handle_request(user_id: str, request_id: str):
    logger.info(
        "Processing request",
        extra={
            "user_id": user_id,
            "request_id": request_id,
            "duration_ms": 123
        }
    )
```

### Log Aggregation

```yaml
# config/log_aggregation.yaml
providers:
    application_insights:
        enabled: true
        connection_string: ${APPLICATIONINSIGHTS_CONNECTION_STRING}
        sample_rate: 0.1 # 10% sampling
        min_level: INFO

    elk_stack:
        enabled: true
        elasticsearch:
            hosts:
                - "elasticsearch:9200"
        logstash:
            host: "logstash"
            port: 5000
        kibana:
            url: "http://kibana:5601"

log_levels:
    production: WARNING
    staging: INFO
    development: DEBUG

retention:
    production: 30 # days
    staging: 7
    development: 1
```

---

## Alerting & Dashboards

### Alert Rules

```yaml
# config/alert_rules.yaml
alerts:
    high_error_rate:
        condition: error_rate > 5%
        duration: 5m
        severity: critical
        notification: slack

    slow_api:
        condition: p95_latency > 1000ms
        duration: 10m
        severity: warning
        notification: email

    db_connection_saturation:
        condition: active_connections > connection_limit * 0.8
        duration: 5m
        severity: warning
        notification: slack

    model_inference_timeout:
        condition: timeout_rate > 1%
        duration: 5m
        severity: high
        notification: pagerduty

notification_channels:
    slack:
        webhook_url: ${SLACK_WEBHOOK}
    email:
        recipients:
            - devops@aria.example.com
    pagerduty:
        integration_key: ${PAGERDUTY_KEY}
```

### Prometheus + Grafana

```bash
#!/bin/bash
# docker-compose.observability.yml

version: '3.8'
services:
  prometheus:
    image: prom/prometheus:latest
    volumes:
      - ./config/prometheus.yml:/etc/prometheus/prometheus.yml
      - prometheus_data:/prometheus
    ports:
      - "9090:9090"
    command:
      - "--config.file=/etc/prometheus/prometheus.yml"

  grafana:
    image: grafana/grafana:latest
    environment:
      GF_SECURITY_ADMIN_PASSWORD: admin
    volumes:
      - grafana_data:/var/lib/grafana
      - ./dashboards:/etc/grafana/provisioning/dashboards
    ports:
      - "3000:3000"

  jaeger:
    image: jaegertracing/all-in-one:latest
    ports:
      - "6831:6831/udp"
      - "16686:16686"
```
